In [ ]:
using MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4
κ_a = 0

In [ ]:
t_max = 30/κ_b
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
re_b_0, im_b_0 = 0, 0
ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0);

In [ ]:
ϵ_p_list = vcat(0.2:.01:1.1)
ϵ_d_g2_ratio_list = 5:.1:13.5

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))
drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_w = 15/κ_b
for i in ProgressBar(1:length(ϵ_p_list))
    ϵ_p_i = ϵ_p_list[i]
    drive.ϵ_0 = ϵ_p_i
    g2_i = besselj(1,ϵ_p_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_i
        drive.ϵ_d = ϵ_d_j
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, u_0_new, p)
        lyap_1 = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
        #lyap_2 = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w+t_max)
        Lyap_coef_chaos[j,i] = lyap_1#(lyap_1*lyap_2 > 0 ? lyap_1 : NaN)
    end
end

In [ ]:
Lyap_coef_chaos_sign = map(x -> x>0 ? 1 : -1, Lyap_coef_chaos);
Lyap_coef_chaos_half_sign = map(x -> x>0 ? x : -1, Lyap_coef_chaos);

In [ ]:
using JSON

ϵ_p_zero_list_Hill_unstable = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/epsilon_zero_list_07_04_2026_Hill_unstable.json");
α_sq_Hill = range(5,13.5,20)./2;

In [ ]:
α_sq= (ϵ_d_g2_ratio_list./2)
g2_list = E_J.*besselj.(1,ϵ_p_list).*φ_a^2 .*φ_b
#g2_zero_list_Hill = E_J.*besselj.(1,ϵ_p_zero_list_Hill).*φ_a^2 .*φ_b
g2_zero_list_Hill_unstable = E_J.*besselj.(1,ϵ_p_zero_list_Hill_unstable).*φ_a^2 .*φ_b
heatmap(g2_list/2π*1e3,α_sq, Lyap_coef_chaos_sign, cmap=:bluesreds)
plot!(g2_zero_list_Hill_unstable/2π*1e3, α_sq_Hill[end:-1:1], label = "Hill's unstability")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\varepsilon_d/2g_2",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2π*1e3),maximum(g2_list/2π*1e3)),ylims=(minimum(α_sq),maximum(α_sq)+2),
    colorbar_title="\n\nSign of Lyapunov exp.",
    clims=(minimum(Lyap_coef_chaos_sign),maximum(Lyap_coef_chaos_sign)),
    right_margin=10mm)
#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/satellites/heatmap_comparison_instability_lyap_origin_start.pdf")

In [ ]:
using JSON
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/data_lyapunov_07_04_2026.json", "w") do io
    JSON.print(io, Lyap_coef_chaos_sign)
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/g2_list_07_04_2026.json", "w") do io
    JSON.print(io, g2_list)
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/alpha_sq_list_07_04_2026.json", "w") do io
    JSON.print(io, α_sq)
end

In [ ]:
ϵ_p = 0.9
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2

t_max=30/κ_b
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a, q_CL_a, t_list_CL, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

# Poincaré sections

In [ ]:
t_max= 30/κ_b
t_min = 0.0

re_a_0, im_a_0 = 2, 2
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

ϵ_p = 0.3
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2
system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

tspan = (t_min, t_max) # Time span for the simulation

#preparation
system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
dt_save=2pi/ω_a
t_save = t_max-30/κ_b:dt_save:t_max 
u_0_tot = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    
ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = default_solver(u_0_tot, tspan, p, dt; equation=default_equation!, t_save = t_save)

In [ ]:
besselj(1,0.9)*E_J*φ_a^2*φ_b/2/pi*1e3

In [ ]:
ϵ_p_list = [0.58, 0.6894736842105263, 0.8263157894736842]#range(0.58, 1.1, 20)#

t_max = 30/κ_b
t_min = 0.0
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(30*sqrt(2)*ω_a)
t_min_save = t_max - 1000*dt_save
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_a_cat = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b_cat = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

ϵ_p = ϵ_p_list[1]
Im_a_list=range(-4,4,3)
Re_a_list=range(-10, 10 , 3)
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
drive_prepared = prepare(drive)
Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_a_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = (re_i + 1im*im_j)

        system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
        system_prepared = prepare(system)

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

        a_rot = a.*exp.(1im*ω_a*t_list)*1im*φ_a #90° rotation
        b_rot = b.*exp.(1im*ω_b*t_list)*1im*φ_b #90° rotation

        if abs(maximum(imag(a_rot)))>0.1
            Final_mode_a[(re_i,im_j)] = a_rot
            Final_mode_b[(re_i,im_j)] = b_rot
        else 
            Final_mode_a_cat[(re_i,im_j)] = a_rot
            Final_mode_b_cat[(re_i,im_j)] = b_rot
        end    
    end
end
Final_all_a[ϵ_p] = Final_mode_a
Final_all_b[ϵ_p] = Final_mode_b
Final_all_a_cat[ϵ_p] = Final_mode_a_cat
Final_all_b_cat[ϵ_p] = Final_mode_b_cat

ϵ_p = ϵ_p_list[2]
Im_a_list=range(-4,4,3)
Re_a_list=range(-10, 14 , 4)
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
drive_prepared = prepare(drive)
Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_a_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = (re_i + 1im*im_j)

        system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
        system_prepared = prepare(system)

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

        a_rot = a.*exp.(1im*ω_a*t_list)*1im*φ_a #90° rotation
        b_rot = b.*exp.(1im*ω_b*t_list)*1im*φ_b #90° rotation

        if abs(maximum(imag(a_rot)))>0.1
            Final_mode_a[(re_i,im_j)] = a_rot
            Final_mode_b[(re_i,im_j)] = b_rot
        else 
            Final_mode_a_cat[(re_i,im_j)] = a_rot
            Final_mode_b_cat[(re_i,im_j)] = b_rot
        end    
    end
end
Final_all_a[ϵ_p] = Final_mode_a
Final_all_b[ϵ_p] = Final_mode_b
Final_all_a_cat[ϵ_p] = Final_mode_a_cat
Final_all_b_cat[ϵ_p] = Final_mode_b_cat

ϵ_p = ϵ_p_list[3]
Im_a_list=range(-4, 4, 10)
Re_a_list=range(-5, 5 , 10)
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
drive_prepared = prepare(drive)
Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_a_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = (re_i + 1im*im_j)

        system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
        system_prepared = prepare(system)

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

        a_rot = a.*exp.(1im*ω_a*t_list)*1im*φ_a #90° rotation
        b_rot = b.*exp.(1im*ω_b*t_list)*1im*φ_b #90° rotation

        Final_mode_a[(re_i,im_j)] = a_rot
        Final_mode_b[(re_i,im_j)] = b_rot
    end
end
Final_all_a[ϵ_p] = Final_mode_a
Final_all_b[ϵ_p] = Final_mode_b
Final_all_a_cat[ϵ_p] = Final_mode_a_cat;
Final_all_b_cat[ϵ_p] = Final_mode_b_cat;

In [ ]:
[besselj(1,ϵ_p)*E_J*φ_a^2*φ_b for ϵ_p in ϵ_p_list]/2pi*1e3

In [ ]:
for ϵ_p in ϵ_p_list
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_a[ϵ_p]
    Final_mode_cat = Final_all_a_cat[ϵ_p]
    p0 = plot()

    # Overlay scatter points
   for key in keys(Final_mode)
        z = Final_mode[key]
        scatter!(p0, real.(z), imag.(z), ms = .5, color = RGBA(185/256,21/256,21/256,0), label = "",)
    end
    for key in keys(Final_mode_cat)
        z = Final_mode_cat[key]
        scatter!(p0, real.(z), imag.(z), ms = .5, color = RGBA(13/256,76/256,116/256,0), label = "",)
    end

    # Add labels and title
    xlims!((-0.75, 0.75))
    ylims!((-0.75, 0.75))
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          top_margin=5mm,
          aspect_ratio = 1,)
    display(p0)
    savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/poster_chicago/poincare_section_07_04_2026_$ϵ_p.jpeg")
end


In [ ]:
ϵ_p_list = range(0.58, 1.1, 40)#[0.58, 0.6894736842105263, 0.8263157894736842]#

t_max = 30/κ_b
t_min = 0.0
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(30*sqrt(2)*ω_a)
t_min_save = t_max - 1000*dt_save
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

Im_a_list=range(-4,4,10)
Re_a_list=range(-2.5, 2.5 ,25)
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_a_cat = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b_cat = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_p in ϵ_p_list
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_a_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = (re_i + 1im*im_j)

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            a_rot = a.*exp.(1im*ω_a*t_list)*1im*φ_a #90° rotation
            b_rot = b.*exp.(1im*ω_b*t_list)*1im*φ_b #90° rotation

            if abs(maximum(real(a_rot)))<0.2 && abs(minimum(real(a_rot)))>0.15 && maximum(abs.(imag(a_rot)))<0.05
                Final_mode_a_cat[(re_i,im_j)] = a_rot
                Final_mode_b_cat[(re_i,im_j)] = b_rot
            else 
                Final_mode_a[(re_i,im_j)] = a_rot
                Final_mode_b[(re_i,im_j)] = b_rot
            end    
        end
    end
    Final_all_a[ϵ_p] = Final_mode_a
    Final_all_b[ϵ_p] = Final_mode_b
    Final_all_a_cat[ϵ_p] = Final_mode_a_cat
    Final_all_b_cat[ϵ_p] = Final_mode_b_cat
end;

In [ ]:
for ϵ_p in ϵ_p_list
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_a[ϵ_p]
    Final_mode_cat = Final_all_a_cat[ϵ_p]
    p0 = plot()

    # Overlay scatter points
    for key in keys(Final_mode_cat)
        z = Final_mode_cat[key]
        scatter!(p0, real.(z), imag.(z), ms = .5, color = RGBA(13/256,76/256,116/256,0), label = "",)
    end

   for key in keys(Final_mode)
        z = Final_mode[key]
        scatter!(p0, real.(z), imag.(z), ms = .5, color = RGBA(185/256,21/256,21/256,0), label = "",)
    end

    # Add labels and title
    xlims!((-0.75, 0.75))
    ylims!((-0.75, 0.75))
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(trunc(Int, g2/2pi*1000))*" MHz",
          top_margin=5mm,
          aspect_ratio = 1,)
    #display(p0)
    savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/poster_chicago/poincare_section_09_04_2026_$ϵ_p.png")
end
